# ae_picker v2 — New Features Demo

This notebook demonstrates every new feature added in **v2**:

| Section | What's new |
|---------|------------|
| 1 | `suggest_parameters` — frequency-adaptive presets |
| 2 | `prepend_noise_aic_picker` — AIC on onset-aligned records |
| 3 | `envelope_onset_picker` + `envelope_offset_picker` — event duration |
| 4 | `plot_section` — wiggle section plot |
| 5 | `flag_outlier_picks` — inter-trace QC |
| 6 | `read_segy` / `read_mseed` — ObsPy-backed readers |
| 7 | `run()` with `profile` and `reader_fn` — batch with presets & custom readers |

All examples use **synthetic waveforms** so no data files are required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ae_picker
from ae_picker import (
    suggest_parameters,
    prepend_noise_aic_picker,
    envelope_onset_picker, envelope_offset_picker,
    aic_picker,
    plot_section,
    flag_outlier_picks,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

---
## 1 · `suggest_parameters` — frequency-adaptive presets

When moving between acquisition setups (AE at ~8 MHz, PS-log at ~4 kHz, refraction at ~500 Hz)
the STA/LTA windows need to scale with the sampling rate.  
`suggest_parameters` does that automatically.

In [ ]:
# --- Auto-detect profile from sampling rate ---
for fs, label in [(8e6, 'AE  (8 MHz)'),
                  (4000, 'PS-log  (4 kHz)'),
                  (500,  'Refraction  (500 Hz)')]:
    p = suggest_parameters(fs)
    print(f"{label:25s}  profile={p['profile']:10s}  "
          f"sta={p['sta_s']*1e3:.3f} ms  lta={p['lta_s']*1e3:.3f} ms")

In [ ]:
# --- Force a profile explicitly ---
p_ref = suggest_parameters(fs=100, profile='refraction')
print("Explicit refraction profile at 100 Hz:")
for k, v in p_ref.items():
    print(f"  {k:30s}: {v}")

In [ ]:
# --- Override STA/LTA windows via dominant frequency ---
# Rule of thumb: sta = 2 / f_dominant,  lta = 20 / f_dominant
p_f = suggest_parameters(fs=4000, f_dominant=200)   # 200 Hz dominant wave
print(f"f_dominant=200 Hz → sta_s={p_f['sta_s']*1e3:.1f} ms  lta_s={p_f['lta_s']*1e3:.1f} ms")

assert abs(p_f['sta_s'] - 2/200) < 1e-12
assert abs(p_f['lta_s'] - 20/200) < 1e-12
print("Assertions passed.")

In [ ]:
# --- Use preset values directly in a picker ---
from ae_picker import refined_stalta_picker

fs = 4000
dt = 1.0 / fs
N  = 2000
t  = np.arange(N) * dt

# Synthetic waveform: noise then a sine burst at t=0.2 s
rng = np.random.default_rng(42)
amp = rng.normal(0, 0.05, N)
onset_s = 0.20
i_on = int(onset_s * fs)
burst = np.sin(2 * np.pi * 200 * t[i_on:i_on+100]) * np.exp(-np.arange(100) / 30)
amp[i_on:i_on+100] += burst

params = suggest_parameters(fs=fs, f_dominant=200)

pick_idx, cft, triggers = refined_stalta_picker(
    amp, t,
    sta_s=params['sta_s'],
    lta_s=params['lta_s'],
)

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t, amp, lw=0.7)
axes[1].plot(t, cft, lw=0.7, color='steelblue', label='STA/LTA')
if pick_idx is not None:
    for ax in axes:
        ax.axvline(t[pick_idx], color='crimson', lw=1.5, label=f'Pick {t[pick_idx]*1e3:.1f} ms')
axes[0].axvline(onset_s, color='limegreen', lw=1.2, ls='--', label='True onset')
axes[0].set_ylabel('Amplitude'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('STA/LTA'); axes[1].set_xlabel('Time (s)'); axes[1].legend(fontsize=8)
axes[0].set_title('suggest_parameters → refined_stalta_picker (PS-log synthetic)')
plt.tight_layout()

---
## 2 · `prepend_noise_aic_picker` — AIC on onset-aligned records

When the record **starts right at the P arrival** (unfiltered AE format), the standard AIC picker
has almost no pre-onset samples and its minimum drifts to the boundary.  
`prepend_noise_aic_picker` fixes this by prepending synthetic Gaussian noise matched
to the record's own noise level.

In [ ]:
# Synthetic onset-aligned record at 200 kHz (5 µs / sample)
fs_ae = 200_000
dt_ae = 1.0 / fs_ae
N_ae  = 500
t_ae  = np.arange(N_ae) * dt_ae   # 0 … 2.5 ms

rng  = np.random.default_rng(7)
noise_rms = 0.02
amp_ae = rng.normal(0, noise_rms, N_ae)

# Arrival starts at sample 8 (40 µs)
true_onset = 8
carrier = np.sin(2 * np.pi * 40_000 * t_ae[true_onset:true_onset+80])
envelope_shape = np.exp(-np.arange(80) / 15.0)
amp_ae[true_onset:true_onset+80] += 0.8 * carrier * envelope_shape

# --- Standard AIC picker (very narrow search window for onset-aligned data) ---
p_std, aic_std = aic_picker(amp_ae, search_start=1, search_end=15)

# --- prepend_noise_aic_picker ---
p_pre, aic_pad, n_pre = prepend_noise_aic_picker(
    amp_ae, t_ae,
    prepend_duration_s=50e-6,   # prepend 50 µs of synthetic noise
)

print(f"True onset sample : {true_onset}  ({t_ae[true_onset]*1e6:.1f} µs)")
print(f"Standard AIC pick : {p_std}  ({t_ae[p_std]*1e6:.1f} µs)")
print(f"Prepend AIC pick  : {p_pre}  ({t_ae[p_pre]*1e6:.1f} µs)")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)

t_us = t_ae * 1e6
axes[0].plot(t_us, amp_ae, lw=0.8, color='steelblue')
axes[0].axvline(t_us[true_onset], color='limegreen', lw=1.5, ls='--', label='True onset')
axes[0].axvline(t_us[p_std], color='crimson', lw=1.5, label=f'Standard AIC ({t_us[p_std]:.1f} µs)')
axes[0].axvline(t_us[p_pre], color='darkorange', lw=1.5, ls='-.', label=f'Prepend AIC ({t_us[p_pre]:.1f} µs)')
axes[0].set_ylabel('Amplitude')
axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_title('Onset-aligned synthetic AE record')
axes[0].set_xlim(0, 200)

axes[1].plot(t_us, aic_std, lw=0.8, color='crimson')
axes[1].axvline(t_us[p_std], color='crimson', lw=1.2, ls='--')
axes[1].set_ylabel('AIC'); axes[1].set_title('Standard AIC function')

# AIC padded: x-axis is in prepended samples; map back to original time
t_pad_us = (np.arange(len(aic_pad)) - n_pre) * dt_ae * 1e6
axes[2].plot(t_pad_us, aic_pad, lw=0.8, color='darkorange')
axes[2].axvline(t_us[p_pre], color='darkorange', lw=1.2, ls='--')
axes[2].axvspan(-50, 0, alpha=0.10, color='gray', label='Prepended noise')
axes[2].set_ylabel('AIC'); axes[2].set_xlabel('Time [µs]')
axes[2].set_title(f'AIC on padded array (n_prepend={n_pre} samples)')
axes[2].legend(fontsize=8)

plt.tight_layout()

---
## 3 · `envelope_onset_picker` + `envelope_offset_picker` — event duration

`envelope_onset_picker` detects the first arrival using Hilbert-envelope hysteresis  
(MAD noise statistics + persistence gating).  
`envelope_offset_picker` then walks forward from the onset to find when the event ends.

In [ ]:
# Synthetic record: noise floor, then an AE burst, then noise again
fs_s = 1_000_000   # 1 MHz
dt_s = 1.0 / fs_s
N_s  = 3000
t_s  = np.arange(N_s) * dt_s      # 0 … 3 ms

rng2  = np.random.default_rng(99)
amp_s = rng2.normal(0, 0.03, N_s)

# Burst: t = 0.5–1.3 ms
i_onset  = int(0.5e-3 * fs_s)
i_offset = int(1.3e-3 * fs_s)
burst_len = i_offset - i_onset
env_shape = np.concatenate([
    np.linspace(0, 1, burst_len // 4),
    np.ones(burst_len // 2),
    np.linspace(1, 0, burst_len - burst_len // 4 - burst_len // 2)
])
carrier_s = np.sin(2 * np.pi * 200_000 * t_s[i_onset:i_offset])
amp_s[i_onset:i_offset] += 0.6 * env_shape * carrier_s

# --- onset ---
onset_idx = envelope_onset_picker(
    amp_s, t_s,
    search_start_s = 0.2e-3,
    search_end_s   = 1.5e-3,
    k_high=4.0, k_low=3.0,
    min_persist_ms=0.05,
    noise_lookback_s=0.3e-3,
)

# --- offset (only if onset was found) ---
offset_idx = None
if onset_idx is not None:
    offset_idx = envelope_offset_picker(
        amp_s, t_s,
        onset_idx=onset_idx,
        search_end_s=2.5e-3,
        k_low=2.5,
        end_persist_ms=0.10,
    )

print(f"True onset   : {i_onset}  ({t_s[i_onset]*1e3:.3f} ms)")
print(f"True offset  : {i_offset}  ({t_s[i_offset]*1e3:.3f} ms)")
print(f"Picked onset : {onset_idx}  ({t_s[onset_idx]*1e3:.3f} ms)" if onset_idx is not None else "Onset: None")
print(f"Picked offset: {offset_idx}  ({t_s[offset_idx]*1e3:.3f} ms)" if offset_idx is not None else "Offset: None")

if onset_idx and offset_idx:
    duration_ms = (t_s[offset_idx] - t_s[onset_idx]) * 1e3
    print(f"Event duration: {duration_ms:.3f} ms  (true: {(t_s[i_offset]-t_s[i_onset])*1e3:.3f} ms)")

In [ ]:
from scipy.signal import hilbert as _hilbert
env = np.abs(_hilbert(amp_s))

t_ms = t_s * 1e3
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

axes[0].plot(t_ms, amp_s, lw=0.6, color='steelblue', label='Waveform')
axes[1].plot(t_ms, env, lw=0.8, color='darkorange', label='Hilbert envelope')

for ax in axes:
    ax.axvline(t_ms[i_onset],  color='limegreen', lw=1.5, ls='--', label='True onset')
    ax.axvline(t_ms[i_offset], color='limegreen', lw=1.5, ls=':',  label='True offset')
    if onset_idx:
        ax.axvline(t_ms[onset_idx],  color='crimson',    lw=1.5, label=f'Picked onset')
    if offset_idx:
        ax.axvline(t_ms[offset_idx], color='royalblue',  lw=1.5, label=f'Picked offset')
    ax.legend(fontsize=7, loc='upper right', ncol=2)
    ax.grid(True, ls=':', alpha=0.4)

axes[0].set_ylabel('Amplitude'); axes[0].set_title('Envelope onset + offset on synthetic burst')
axes[1].set_ylabel('Envelope'); axes[1].set_xlabel('Time (ms)')
plt.tight_layout()

---
## 4 · `plot_section` — wiggle section plot

Visualises a suite of traces as a **wiggle section** (SEG convention: positive lobes filled).  
Supports refraction layout (`time_axis='x'`) and PS-log / VSP layout (`time_axis='y'`).

In [ ]:
# Synthetic refraction survey: 12 receivers at 10–120 m offset
# V_apparent = 1200 m/s; dominant frequency ≈ 40 Hz at 1 kHz sampling
n_traces  = 12
offsets_m = np.arange(1, n_traces + 1) * 10.0   # 10, 20, … 120 m
V_app     = 1200.0   # apparent velocity m/s
t0_static = 0.005    # intercept time 5 ms

fs_r  = 1000.0
dt_r  = 1.0 / fs_r
N_r   = 400
t_r   = np.arange(N_r) * dt_r

rng3 = np.random.default_rng(0)
channels_r = []
true_picks = []

for off in offsets_m:
    tp = t0_static + off / V_app       # linear moveout
    true_picks.append(tp)
    ip = int(tp * fs_r)
    a  = rng3.normal(0, 0.04, N_r)
    if ip < N_r - 60:
        f_dom = 40.0
        win   = np.zeros(60)
        win[:30] = np.sin(2 * np.pi * f_dom * np.arange(30) * dt_r)
        win[30:] = -np.sin(2 * np.pi * f_dom * np.arange(30) * dt_r) * 0.5
        a[ip:ip+60] += win
    channels_r.append({'time': t_r.copy(), 'amp': a})

# --- run AIC on each trace ---
aic_picks = []
for ch in channels_r:
    ip_true = int(true_picks[channels_r.index(ch)] * fs_r)
    ss = max(1, ip_true - 15)
    se = min(N_r - 2, ip_true + 20)
    idx, _ = aic_picker(ch['amp'], search_start=ss, search_end=se)
    aic_picks.append(t_r[idx])

# --- refraction layout (time horizontal) ---
fig = plot_section(
    channels_r,
    picks={'True moveout': true_picks, 'AIC': aic_picks},
    offsets_m=offsets_m,
    time_axis='x',
    normalise='trace',
)
fig.suptitle('Refraction wiggle section (time_axis="x")', y=1.01, fontsize=11)
plt.show()

In [ ]:
# --- PS-log / VSP layout (time vertical, depth on x-axis) ---
depths_m = np.arange(50, 50 + n_traces * 10, 10, dtype=float)   # 50…160 m depth

fig = plot_section(
    channels_r,
    picks={'AIC': aic_picks},
    offsets_m=depths_m,
    time_axis='y',
    normalise='global',
)
fig.suptitle('PS-log wiggle section (time_axis="y")', y=1.01, fontsize=11)
plt.show()

---
## 5 · `flag_outlier_picks` — inter-trace QC

Fits a linear (or robust Theil–Sen) travel-time model through the picks and flags any pick
whose residual exceeds `n_mad` × MAD.  NaN picks are always flagged.

In [ ]:
# Synthetic picks: 10 traces on a 1 ms / 10 m moveout; last trace is a cycle-skip
offsets = np.arange(10, 110, 10, dtype=float)   # 10…100 m
V_true  = 1000.0   # m/s  →  1 ms / 10 m
picks_s = offsets / V_true * 1e-3 + 0.005       # intercept 5 ms

# Add small Gaussian jitter to clean picks
rng4 = np.random.default_rng(11)
picks_s += rng4.normal(0, 0.0003, len(picks_s))

# Inject one cycle-skip (trace 7, index 6) and one NaN (trace 9, index 8)
picks_s[6] += 0.015   # large outlier
picks_s[8]  = np.nan  # missing pick

df_picks = pd.DataFrame({'aic_pick_s': picks_s})

# --- linear model ---
outliers_lin, model_lin = flag_outlier_picks(
    df_picks, offsets, 'aic_pick_s', n_mad=3.0, model='linear', return_model=True
)

print("Linear model:")
print(f"  slope       = {model_lin['slope']*1e3:.4f} ms/m")
print(f"  velocity    = {model_lin['velocity_ms']:.1f} m/s")
print(f"  intercept   = {model_lin['intercept']*1e3:.3f} ms")
print(f"\nOutlier mask: {outliers_lin.values}")
print(f"Flagged indices: {list(outliers_lin[outliers_lin].index)}")

In [ ]:
# Compare linear vs robust (Theil-Sen) flagging
try:
    outliers_rob, model_rob = flag_outlier_picks(
        df_picks, offsets, 'aic_pick_s', n_mad=3.0, model='robust', return_model=True
    )
    print("Robust (Theil-Sen) model:")
    print(f"  slope       = {model_rob['slope']*1e3:.4f} ms/m")
    print(f"  velocity    = {model_rob['velocity_ms']:.1f} m/s")
    has_robust = True
except ImportError as e:
    print(f"SciPy not available: {e}")
    outliers_rob = outliers_lin
    model_rob = model_lin
    has_robust = False

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

# All picks
valid = ~outliers_lin
ax.scatter(offsets[valid.values], picks_s[valid.values] * 1e3,
           color='steelblue', s=60, zorder=4, label='Clean picks')
ax.scatter(offsets[outliers_lin.values], picks_s[outliers_lin.values] * 1e3,
           color='crimson', marker='x', s=120, linewidths=2, zorder=5, label='Flagged (linear)')

# Model lines
x_line = np.linspace(0, 110, 200)
y_lin  = (model_lin['slope'] * x_line + model_lin['intercept']) * 1e3
ax.plot(x_line, y_lin, 'k--', lw=1.2, label=f"Linear fit  V={model_lin['velocity_ms']:.0f} m/s")

if has_robust:
    y_rob = (model_rob['slope'] * x_line + model_rob['intercept']) * 1e3
    ax.plot(x_line, y_rob, 'darkorange', lw=1.2, ls='-.', label=f"Robust fit  V={model_rob['velocity_ms']:.0f} m/s")

ax.set_xlabel('Offset (m)')
ax.set_ylabel('Pick time (ms)')
ax.set_title('flag_outlier_picks — synthetic moveout with cycle-skip and NaN')
ax.legend(fontsize=8)
ax.grid(True, ls=':', alpha=0.4)
plt.tight_layout()

---
## 6 · `read_segy` / `read_mseed` — ObsPy-backed readers

These are optional readers that require **ObsPy**.  
Install it with `pip install ae-picker[obspy]` (or `pip install obspy`).

The cells below demonstrate the API by creating a minimal synthetic SEG-Y file in memory
using ObsPy, then reading it back through `read_segy`.  
If ObsPy is not installed the cells will show a clear `ImportError`.

In [ ]:
import io as _io, tempfile, os

try:
    from obspy import Trace, Stream
    from obspy import UTCDateTime
    from ae_picker import read_segy, read_mseed
    OBSPY_AVAILABLE = True
    print("ObsPy is available.")
except ImportError:
    OBSPY_AVAILABLE = False
    print("ObsPy is NOT installed.  Install with: pip install ae-picker[obspy]")
    print("Skipping read_segy / read_mseed cells.")

In [ ]:
if OBSPY_AVAILABLE:
    # Build a 3-trace miniSEED stream in a temp file, then read it back
    rng5  = np.random.default_rng(55)
    fs_ms = 500.0
    n_ms  = 300

    traces = []
    for i in range(3):
        data = rng5.normal(0, 0.1, n_ms).astype(np.float32)
        onset = 50 + i * 10
        data[onset:onset+50] += (0.5 * np.sin(2*np.pi*20*np.arange(50)/fs_ms) *
                                  np.exp(-np.arange(50)/15)).astype(np.float32)
        tr = Trace(data=data)
        tr.stats.sampling_rate = fs_ms
        tr.stats.network = 'SY'
        tr.stats.station = f'ST{i:02d}'
        tr.stats.channel = 'BHZ'
        traces.append(tr)

    st = Stream(traces)

    with tempfile.NamedTemporaryFile(suffix='.mseed', delete=False) as f:
        tmp_mseed = f.name
    st.write(tmp_mseed, format='MSEED')

    meta_ms, channels_ms = read_mseed(tmp_mseed)
    os.unlink(tmp_mseed)

    print("meta keys :", list(meta_ms.keys()))
    print(f"n_traces  : {meta_ms['n_traces']}")
    print(f"station   : {meta_ms['station']}")
    print(f"channel   : {meta_ms['channel']}")
    print(f"time shape: {channels_ms[0]['time'].shape}")

In [ ]:
if OBSPY_AVAILABLE:
    from ae_picker import aic_picker as _aic

    fig, axes = plt.subplots(len(channels_ms), 1, figsize=(10, 2.5*len(channels_ms)), sharex=True)
    if len(channels_ms) == 1:
        axes = [axes]

    for i, (ch, ax) in enumerate(zip(channels_ms, axes)):
        t_ms2 = ch['time']
        ax.plot(t_ms2, ch['amp'], lw=0.8, color='steelblue')
        N2 = len(t_ms2)
        pick_i, _ = _aic(ch['amp'], search_start=20, search_end=N2//2)
        ax.axvline(t_ms2[pick_i], color='crimson', lw=1.5,
                   label=f'AIC  t={t_ms2[pick_i]*1e3:.1f} ms')
        ax.set_ylabel('Amplitude'); ax.set_title(f'Trace {i+1} (from miniSEED)')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, ls=':', alpha=0.4)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle('read_mseed → aic_picker', fontsize=11, y=1.01)
    plt.tight_layout()
else:
    print("(Skipped — ObsPy not available)")

---
## 7 · `run()` with `profile` and `reader_fn` — batch with presets & custom readers

Two new parameters were added to `batch.run()`:

* **`profile`** — calls `suggest_parameters` internally so STA/LTA windows scale  
  to the data's sampling rate automatically.
* **`reader_fn`** — plug in any custom reader (e.g. for CSV refraction data or ObsPy streams)  
  without touching the batch runner.

The example below writes a tiny in-memory dataset to a temp directory and runs the batch.

In [ ]:
import tempfile, shutil
from pathlib import Path
from ae_picker import run

# --- Create a temporary dataset directory ---
tmpdir = Path(tempfile.mkdtemp())
sig_dir = tmpdir / 'custom_signals'
sig_dir.mkdir()

# Write 3 synthetic CSV files (tab-separated, two-column: time, amplitude)
fs_b = 4000.0
dt_b = 1.0 / fs_b
N_b  = 1000
t_b  = np.arange(N_b) * dt_b
rng6 = np.random.default_rng(77)

for k in range(3):
    a = rng6.normal(0, 0.05, N_b)
    onset = 150 + k * 20
    a[onset:onset+80] += (
        np.sin(2*np.pi*200*t_b[onset:onset+80]) *
        np.exp(-np.arange(80)/20.0)
    )
    df_out = pd.DataFrame({'time_s': t_b, 'amp_V': a})
    df_out.to_csv(sig_dir / f'event_{k:02d}.csv', sep='\t', index=False)

print("Temp files:", [p.name for p in sig_dir.glob('*.csv')])

In [ ]:
# --- Custom reader: wraps the CSV format into the standard (meta, channels) contract ---
def my_csv_reader(filepath):
    df_in = pd.read_csv(filepath, sep='\t')
    meta = {'filename': Path(filepath).stem}
    channels = [{'time': df_in['time_s'].values, 'amp': df_in['amp_V'].values}]
    return meta, channels

# --- Run batch with ps_log profile + custom reader ---
df_batch = run(
    data_dir    = tmpdir,
    signal_dirs = ('custom_signals',),
    output_dir  = tmpdir / 'output',
    pickers     = ('aic', 'refined_stalta'),
    profile     = 'ps_log',          # <-- NEW: STA/LTA windows auto-scaled
    reader_fn   = my_csv_reader,     # <-- NEW: bypass built-in readers
    file_glob   = '*.csv',
    plot        = False,
    save_plots  = False,
    show_plots  = False,
)

print(df_batch[['file', 'channel', 'aic_pick_s', 'refined_stalta_pick_s']].to_string(index=False))

In [ ]:
# Check that the STA/LTA used the ps_log preset values
p_ps = suggest_parameters(fs=fs_b, profile='ps_log')
print(f"ps_log preset → sta_s={p_ps['sta_s']*1e3:.1f} ms  lta_s={p_ps['lta_s']*1e3:.1f} ms")
print("(These were passed to refined_stalta_picker automatically by run())")

# Clean up
shutil.rmtree(tmpdir, ignore_errors=True)
print("Temp directory removed.")

---
### Summary of v2 additions

| Function | Module | What it does |
|---|---|---|
| `suggest_parameters` | `ae_picker.presets` | Auto-scales STA/LTA & picker windows to any sampling rate |
| `prepend_noise_aic_picker` | `ae_picker.pickers` | Stabilises AIC on onset-aligned records by prepending synthetic noise |
| `envelope_onset_picker` | `ae_picker.pickers` | Hilbert-envelope first-arrival with MAD noise stats + persistence |
| `envelope_offset_picker` | `ae_picker.pickers` | Forward walk to detect event end |
| `plot_section` | `ae_picker.plot` | Wiggle section (refraction / PS-log / VSP layouts) |
| `flag_outlier_picks` | `ae_picker.qc` | Inter-trace QC via linear or robust moveout fit |
| `read_segy` / `read_mseed` | `ae_picker.io` | Optional ObsPy-backed readers returning standard `(meta, channels)` |
| `run(profile=…, reader_fn=…)` | `ae_picker.batch` | Batch runner now supports presets and custom readers |